In [1]:
from qiskit.circuit import Parameter, QuantumCircuit, QuantumRegister, ClassicalRegister

from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector, Operator
from scipy.optimize import minimize 
from qiskit.circuit.library import QFT
from qiskit import transpile
from qiskit.circuit.library import UnitaryGate

import random
import matplotlib.pyplot as plt
import scipy.linalg as scl
import numpy as np
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()

# backend=FakeKyiv()
# sampler = Sampler(backend = backend)
pm = generate_preset_pass_manager(backend=backend,optimization_level=2)

In [2]:


nb_qubits = 3

N = 2**nb_qubits
m = np.zeros((N,N))
for j in range(N):
    if j == N-1:
        break
    else:
       m[j,j+1] = -1 

for j in range(N):
    if j == N-1:
        break
    else:
       m[j+1,j] = -1 
for j in range(N):
   m[j,j] = 2 
m[0] = np.array([1]+ [0]*(N-1))
m[1,0] = 0

b = np.array([0,0.25,0.25,0.25,0.25,0.5,0.5,0.5])
m

array([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  2., -1.,  0.,  0.,  0.,  0.,  0.],
       [ 0., -1.,  2., -1.,  0.,  0.,  0.,  0.],
       [ 0.,  0., -1.,  2., -1.,  0.,  0.,  0.],
       [ 0.,  0.,  0., -1.,  2., -1.,  0.,  0.],
       [ 0.,  0.,  0.,  0., -1.,  2., -1.,  0.],
       [ 0.,  0.,  0.,  0.,  0., -1.,  2., -1.],
       [ 0.,  0.,  0.,  0.,  0.,  0., -1.,  2.]])

In [3]:
d,u = np.linalg.eig(m)
k = np.max(d)/np.min(d)
dm = max(d)
# m = m/dm

In [4]:
np.linalg.det(m)



7.999999999999998

In [5]:
def U_b(nb_qubits):
    circ = QuantumCircuit(nb_qubits)
    circ.prepare_state(b)
    return circ
U = U_b(nb_qubits)
b = np.array(Statevector(U_b(nb_qubits)))
b

array([1.80411242e-16+6.93889390e-17j, 2.50000000e-01+5.13478149e-16j,
       2.50000000e-01-2.98372438e-16j, 2.50000000e-01-3.33066907e-16j,
       2.50000000e-01-6.66133815e-16j, 5.00000000e-01-6.10622664e-16j,
       5.00000000e-01+6.80011603e-16j, 5.00000000e-01+7.21644966e-16j])

In [6]:
def Hamiltonian(m):
    Ub = np.array(Operator(U_b(nb_qubits)))
    z = np.array([[1,0],
                 [0,-1]])
    I = np.array([[1,0],
                 [0,1]])
    def tensor(j,k,l):
        return np.kron(j,np.kron(k,l))
    M1 = (np.dot(np.dot(Ub,tensor(z,I,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,z,I)),np.conj(Ub.T))
          +np.dot(np.dot(Ub,tensor(I,I,z)),np.conj(Ub.T)))
    M = 0.5*np.dot(np.dot(np.conj(m.T),(tensor(I,I,I) - M1/nb_qubits)),m)

    return M
A = Hamiltonian(m)

In [7]:
U1=scl.expm(2**0*2*np.pi*1j*A) 
U2=scl.expm(2**1*2*np.pi*1j*A) 
U3=scl.expm(2**2*2*np.pi*1j*A) 
U4=scl.expm(2**3*2*np.pi*1j*A) 
U5=scl.expm(2**4*2*np.pi*1j*A) 
U6=scl.expm(2**5*2*np.pi*1j*A) 
U7=scl.expm(2**6*2*np.pi*1j*A) 
U8=scl.expm(2**7*2*np.pi*1j*A)
 
u1gate = UnitaryGate(U1)
u2gate = UnitaryGate(U2)
u3gate=UnitaryGate(U3)
u4gate=UnitaryGate(U4)
u5gate=UnitaryGate(U5)
u6gate=UnitaryGate(U6)
u7gate=UnitaryGate(U7)
u8gate=UnitaryGate(U8)


C_u1gate=u1gate.control()
C_u2gate=u2gate.control()
C_u3gate=u3gate.control()
C_u4gate=u4gate.control()
C_u5gate=u5gate.control()
C_u6gate=u6gate.control()
C_u7gate=u7gate.control()
C_u8gate=u8gate.control()

In [8]:
x_exact = np.linalg.solve(m,b)
x_exact = x_exact/np.linalg.norm(x_exact)
nb_qubits = 3
depth = 2
qubits = list(range(nb_qubits))
N = len(qubits)
nb_params = int(9*N*depth)

Parameters = np.array([random.random() for _ in range(0, nb_params)])
Parameters

array([0.48030172, 0.60986829, 0.49543928, 0.0872683 , 0.02624919,
       0.0726779 , 0.10737257, 0.77060078, 0.57869663, 0.11079706,
       0.05964047, 0.37159349, 0.25513713, 0.59154793, 0.10729027,
       0.19172419, 0.68601801, 0.65122192, 0.35449546, 0.13802462,
       0.39082822, 0.29289774, 0.66677597, 0.07251441, 0.27873136,
       0.02466956, 0.63844115, 0.47285169, 0.38078124, 0.34401854,
       0.27040991, 0.88128749, 0.4738949 , 0.95362418, 0.82292702,
       0.18342863, 0.99359525, 0.44571499, 0.96447826, 0.28422787,
       0.4711034 , 0.03291901, 0.70979016, 0.54258664, 0.83406145,
       0.36702619, 0.95194409, 0.48052063, 0.00764395, 0.87293303,
       0.1220922 , 0.32042267, 0.01500664, 0.67930908])

In [9]:
RMSE = []
Shots = [100, 1000, 10000, 100000]
for shots in Shots:
    Parameters = np.array([0.72929026, 0.30288237, 0.46143815, 0.26596671, 0.67183524,
       0.54130532, 0.81915447, 0.72206408, 0.81700933, 0.06567133,
       0.4264614 , 0.26218765, 0.30836699, 0.84206223, 0.46043545,
       0.76388847, 0.65271977, 0.43189865, 0.94377196, 0.25803797,
       0.70624087, 0.42226178, 0.35667001, 0.62010337, 0.96449197,
       0.42527427, 0.73003607, 0.15631641, 0.11704865, 0.75895554,
       0.14600023, 0.62478681, 0.40239605, 0.67915165, 0.46055496,
       0.07233959, 0.75761447, 0.59368671, 0.49389645, 0.92579763,
       0.7929723 , 0.91211546, 0.50267549, 0.29768071, 0.89068361,
       0.07732688, 0.49061991, 0.86191129, 0.59055312, 0.6784743 ,
       0.51870635, 0.74901173, 0.67103851, 0.01526466])
    
    def ansatz(Parameters):
        qc = QuantumCircuit(N)
        for d in range(depth):
            param1=Parameters[d*9*N:(d+1)*(9*N)]
            for q in range(N):
                qc.ry(param1[q],qubits[q])
                qc.ry(param1[q+N],qubits[q])
                qc.ry(param1[q+2*N],qubits[q])
            qc.barrier()
            for q in range(N):
                qc.cx(qubits[q], qubits[(q+1)% N])
                qc.ry(param1[q+3*N],qubits[q])
                qc.ry(param1[q+4*N],qubits[(q+1)% N])
                qc.cx(qubits[(q+1)% N], qubits[q])
                qc.ry(param1[q+5*N],qubits[(q+1)% N])
                qc.cx(qubits[q], qubits[(q+1)% N])
            qc.barrier()
            if d==depth-1:
                for q in range(N):
                    qc.ry(param1[q+6*N],qubits[q])
                    qc.ry(param1[q+7*N],qubits[q])
                    qc.ry(param1[q+8*N],qubits[q])
            qc.barrier()
        
        return qc    
    
    
    def circ(parameters):
        x=QuantumRegister(11)
        c=ClassicalRegister(8)
        circuit = QuantumCircuit(x,c)
        phi=parameters
        circuit=circuit.compose(ansatz(parameters),x[8:11])
        circuit.h(x[0:8]) 
        circuit.append(C_u1gate, [x[0],x[8],x[9],x[10]])    
        circuit.append(C_u2gate, [x[1],x[8],x[9],x[10]])    
        circuit.append(C_u3gate, [x[2],x[8],x[9],x[10]])    
        circuit.append(C_u4gate, [x[3],x[8],x[9],x[10]]) 
        circuit.append(C_u5gate, [x[4],x[8],x[9],x[10]])
        circuit.append(C_u6gate, [x[5],x[8],x[9],x[10]])    
        circuit.append(C_u7gate, [x[6],x[8],x[9],x[10]]) 
        circuit.append(C_u8gate, [x[7],x[8],x[9],x[10]])
        circuit &= QFT(num_qubits = 8, approximation_degree = 0, do_swaps = True, 
                       inverse = True, insert_barriers = False, name='qft')
        circuit.measure(x[0:8],c)       
        return circuit

    def cost(Parameters):
        job = backend.run(pm.run(circ(Parameters)),shots = shots).result()
        result = job.get_counts(0)
        if '00000000'not in result: 
            res = 1
        else:
            res = 1 - result['00000000']/shots
        return res
    # cost(parameters)

    def Optimizer(fun, x0, args=(), maxfev=None, 
                  reset_interval=None, eps=None, callback=None, **_):
        
        x0 = np.asarray(x0)
        recycle_z0 = None
        niter = 0
        funcalls = 0
    
        while True:
    
            idx = niter % x0.size
    
            if reset_interval > 0:
                if niter % reset_interval == 0:
                    recycle_z0 = None
    
            if recycle_z0 is None:
                z0 = fun(np.copy(x0), *args)
                funcalls += 1
            else:
                z0 = recycle_z0
    
            p = np.copy(x0)
            p[idx] = x0[idx] + np.pi / 2
            z1 = fun(p, *args)
            funcalls += 1
    
            p = np.copy(x0)
            p[idx] = x0[idx] - np.pi / 2
            z3 = fun(p, *args)
            funcalls += 1
    
            z2 = z1 + z3 - z0
            c = (z1 + z3) / 2
            a = np.sqrt((z0 - z2) ** 2 + (z1 - z3) ** 2) / 2
            b = np.arctan((z1 - z3) / ((z0 - z2) + 1e-32 * (z0 == z2))) + x0[idx]
            b += 0.5 * np.pi + 0.5 * np.pi * np.sign((z0 - z2) + eps * (z0 == z2))
            x0[idx] = b
            recycle_z0 = c - a
            if callback is not None:
                callback(np.copy(x0))
            if funcalls >= maxfev:
                break
            niter += 1
        # return OptimizeResult(fun=problabel0(np.copy(x0)), x=x0, nit=niter, 
        #                       nfev=funcalls, success=(niter > 1))
    
    def save(Parameters):
        global Cost,Params
        Cost.append(cost(Parameters))
        Params.append(Parameters)
        # print(cost(Parameters))
    Cost = []
    Params = []
    Optimizer(cost, Parameters, args=(), maxfev = 4000, 
              reset_interval = 32, eps=1e-32, callback=save)


    e = []
    F = []
    norm_e = []
    for k in range(len(Params)):
        state = np.array(Statevector(ansatz(Params[k])))
        norm = np.dot(state,x_exact)
        e.append(x_exact - state/norm)
        f = abs(np.dot(state,x_exact))**2
        F.append(f)

    for v in e:
        norm_e.append(float(np.linalg.norm(v)))
    Res = norm_e[np.argmax(F)]
    print(Res)
    RMSE.append(Res)

/tmp/ipykernel_870194/2759997384.py:58: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit &= QFT(num_qubits = 8, approximation_degree = 0, do_swaps = True,


0.025268870089557714
0.009712802814407678
0.0028662668362718375
0.0009667967818177185


In [10]:
print(RMSE)

[0.025268870089557714, 0.009712802814407678, 0.0028662668362718375, 0.0009667967818177185]


In [ ]:
[0.025268870089557714, 0.009712802814407678, 0.0028662668362718375, 0.0009667967818177185]
